In [8]:
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
from datetime import datetime, timezone
from dateutil.relativedelta import relativedelta

# Use the exact endpoint from your working script
BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"

# 1. Build the Bulletproof Session to prevent Error 10054
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retries)
session.mount("https://", adapter)
session.headers.update({
    "User-Agent": "KalshiQuantScout/3.0",
    "Accept": "application/json",
    "Connection": "keep-alive"
})

def categorize_nba_market(title, subtitle):
    """Categorizes the market based on its text."""
    text_to_check = f"{title} {subtitle}".lower()
    if any(k in text_to_check for k in ["pts", "reb", "ast", "threes", "points", "rebounds", "assists"]):
        return "Player Prop"
    elif "spread" in text_to_check or "+" in subtitle or "-" in subtitle:
        return "Spread"
    elif "total" in text_to_check or "over/under" in text_to_check:
        return "Total (Over/Under)"
    else:
        return "Moneyline / Winner"

def get_start_price(ticker):
    """Fetches the 1-minute candlestick right before the market locked at game time."""
    url = f"{BASE_URL}/historical/markets/{ticker}/candlesticks?period=1&limit=10"
    try:
        response = session.get(url, timeout=5)
        if response.status_code == 200:
            candles = response.json().get("candlesticks", [])
            if candles:
                return candles[-1].get("close")
    except Exception:
        pass
    return None

def fetch_historical_nba_events():
    now = datetime.now(timezone.utc)
    two_years_ago = now - relativedelta(years=2)
    
    params = {
        "limit": 100, 
        "status": "settled", 
        "with_nested_markets": "true"
    }
    
    processed_data = []
    cursor = None
    has_more = True
    
    print(f"🚀 Initiating Event-Based 2-Year NBA Scrape...")
    print(f"Targeting settled events back to {two_years_ago.strftime('%Y-%m-%d')}\n")

    while has_more:
        if cursor: 
            params["cursor"] = cursor
            
        try:
            response = session.get(f"{BASE_URL}/events", params=params, timeout=10)
            if response.status_code != 200:
                print(f"API Error {response.status_code}. Stopping.")
                break
                
            data = response.json()
            events = data.get("events", [])
            cursor = data.get("cursor")
            
            if not events:
                break

            for event in events:
                # Filter for NBA / Sports categories
                category = event.get("category", "").lower()
                if category not in ["sports", "nba", "basketball"]:
                    continue

                # Ensure we only process events related to the NBA
                if "nba" not in event.get("title", "").lower() and "nba" not in event.get("event_ticker", "").lower():
                    continue

                for market in event.get("markets", []):
                    if market.get("status") != "settled":
                        continue
                        
                    # Check the settlement time
                    settled_str = market.get("settled_time")
                    if not settled_str:
                        continue
                        
                    settled_dt = datetime.fromisoformat(settled_str.replace('Z', '+00:00'))
                    
                    # If we have scrolled back past 2 years, we can stop
                    if settled_dt < two_years_ago:
                        has_more = False
                        break

                    raw_result = market.get("result", "").lower()
                    if raw_result not in ["yes", "no"]:
                        continue
                        
                    ticker = market.get("ticker")
                    title = market.get("title", "") or event.get("title", "")
                    subtitle = market.get("subtitle", "") or market.get("yes_sub_title", "")
                    
                    # Fetch start price (Takes time)
                    start_price = get_start_price(ticker)
                    if start_price is None:
                        start_price = market.get("last_price", 0)

                    processed_data.append({
                        "Event Name": event.get("title", ""),
                        "Market Details": f"{title} - {subtitle}",
                        "Market Category": categorize_nba_market(title, subtitle),
                        "Price At Start (Cents)": start_price,
                        "Resolved Correctly (1=Yes, 0=No)": 1 if raw_result == "yes" else 0,
                        "Market Ticker": ticker,
                        "Settled Time": settled_str
                    })
                    
            if not cursor:
                has_more = False
            else:
                print(f"Processed page... Found {len(processed_data)} NBA markets so far.")
                time.sleep(0.5) # Rate limit protection

        except requests.exceptions.ConnectionError:
            print("⚠️ Connection forcibly closed by Kalshi. Re-establishing session...")
            time.sleep(5)
            continue
            
    df = pd.DataFrame(processed_data)
    
    if not df.empty:
        df["Implied Probability"] = df["Price At Start (Cents)"] / 100.0
        df = df.sort_values(by="Settled Time", ascending=False).reset_index(drop=True)
        df.to_csv("kalshi_nba_2yr_history.csv", index=False)
        print(f"\n✅ Complete! Saved {len(df)} historical NBA markets to CSV.")
    else:
        print("\n⚠️ Found 0 markets. Make sure the API is accessible.")
        
    return df

# Execute
nba_df = fetch_historical_nba_events()

🚀 Initiating Event-Based 2-Year NBA Scrape...
Targeting settled events back to 2024-06-07

Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA markets so far.
Processed page... Found 0 NBA marke

KeyboardInterrupt: 

In [6]:

nba_historical_df = fetch_2yr_nba_dataset()
if not nba_historical_df.empty:
    print(nba_historical_df.head(10))

🚀 Initiating 2-Year NBA Market Scrape (Target: 2024-06-07 to Today)...

Fetching series: NBA...

Fetching series: NBAPROPS...

Fetching series: NBASPREAD...

Fetching series: INBA...

⚠️ No markets found matching the criteria.


In [7]:
import requests
import json

# Testing the elections domain with the NBA series
url = "https://api.elections.kalshi.com/trade-api/v2/historical/markets?series_ticker=NBA&limit=5"

headers = {"accept": "application/json"}
response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    markets = data.get("markets", [])
    
    print(f"Total markets returned in test: {len(markets)}\n")
    
    if markets:
        # Print the very first market beautifully so we can inspect the keys
        print(json.dumps(markets[0], indent=4))
    else:
        print("API returned 0 markets for series 'NBA'. We need to test 'KXNBA' or another ticker.")
else:
    print(f"Error {response.status_code}: {response.text}")

Total markets returned in test: 0

API returned 0 markets for series 'NBA'. We need to test 'KXNBA' or another ticker.
